# NLP

## Setup 

This setup allows you to use *Python* and *R* in the same notebook.


In [ ]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [ ]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [15]:
%%R

# My commonly used R imports

require('tidyverse')

In [16]:
from tqdm.notebook import tqdm
tqdm.pandas()

## Load Data & Remove Duplciates 🧹

In [17]:
# read data
stories = pd.read_csv("output/stories_df.csv", 
                 parse_dates=['publication_date', 'capture_time'])


duplicates to delete

In [ ]:
dedupe_by = ['title', 'domain']
stories[stories.duplicated(subset=dedupe_by, keep=False)]\
    .sort_values(by=dedupe_by)\
    .head()

In [ ]:
# remove duplicates
stories.drop_duplicates(subset=dedupe_by, keep='last', inplace=True)

# preview
stories

# Keywords

In [ ]:
from yake import KeywordExtractor
from pandarallel import pandarallel

kw_extractor = KeywordExtractor()

def get_keywords(text):
    keywords = kw_extractor.extract_keywords(text)
    return [x for x,y in keywords]

pandarallel.initialize(progress_bar=True)
stories['keywords'] = stories['snippet'].parallel_apply(get_keywords)

# display
stories

## Embeddings

In [21]:
import os
import openai
import dotenv
dotenv.load_dotenv()

openai.organization = None
openai.api_key = os.getenv("OPENAI_API_KEY")
# openai.Model.list() # see all openai models

In [ ]:
# Import modules
import tiktoken
from openai import OpenAI
client = OpenAI()

# Set embedding model parameters
embedding_model = "text-embedding-3-small" # this is the model we will use to make embeddings
embedding_encoding = "cl100k_base"  # this the encoding for text-embedding-ada-002
max_tokens = 8000  # the maximum for text-embedding-ada-002 is 8191

# Get the encoding for the specified model
encoding = tiktoken.get_encoding(embedding_encoding)

# Make a new column with the combined title and summary
stories["combined"] = (
    "Title: " + stories.title.str.strip() + "; Content: " + stories.snippet.str.strip()
)

# Make a new column with the number of tokens in the combined title and summary
stories["n_tokens"] = stories.combined.apply(lambda x: len(encoding.encode(x)))

# Sort by that column
stories = stories.sort_values(by='n_tokens', ascending=False)

# Display the bills
stories


In [ ]:
# Grab the rows where the text is too big for the context window of the mmodel (>8000 tokens)
too_long = stories.query("n_tokens > @max_tokens") 

# Print how many will be removed
print(f"Removing {len(too_long)} stories that are too long")

# Display the removed stories here in this cell so we can see what we're losing
display(too_long)  

# Remove the rows where the text is too big for the context window of the model
stories = stories.query("n_tokens <= @max_tokens")  

In [ ]:
# define a function to get embeeddings
def get_embedding(text, model="text-embedding-3-small"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input = [text], model=model).data[0].embedding

# Get the embeddings for the combined title and summary
stories['embedding'] = stories.combined.progress_apply(lambda x: get_embedding(x, model='text-embedding-3-small'))

# Save to CSV
stories.to_csv('stories-with-embeddings.csv')

# Display the dataframe with embeddings
stories.head(2)

In [26]:
# drop combined column since those were only for the purposes of making the embeddings
stories = stories.drop(columns=['combined', 'n_tokens'])

## Dimensionality Reduction (t-SNE)


In [27]:
from sklearn.manifold import TSNE
import numpy as np

# check if vis_dims exists
if os.path.exists("data/stories-with-vis-dims.csv"):
    stories = pd.read_csv("data/stories-with-vis-dims.csv")
else: 
    # Convert to a list of lists of floats
    matrix = np.array(stories.embedding.to_list())

    # Create a t-SNE model and transform the data
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='random', learning_rate=400)
    vis_dims = tsne.fit_transform(matrix)

    # add to dataframe and write to csv
    stories = stories\
        .assign(
            x = vis_dims[:,0], 
            y = vis_dims[:,1])


In [ ]:
stories.to_csv('output/stories-with-nlp.csv', index=False)
stories.head()

# Topic Modeling

In [29]:
stories.reset_index(drop=True, inplace=True)

In [ ]:
stories

In [ ]:
from sklearn.cluster import DBSCAN
# Convert embedding to a NumPy array
X = np.stack(stories['embedding'].values)

# Perform DBSCAN clustering
dbscan = DBSCAN(eps=0.5, min_samples=10)  # Adjust eps and min_samples as per your requirement
labels = dbscan.fit_predict(X)

# Assign topics to DataFrame
stories['topic'] = labels

# Group articles by topic
grouped = stories.groupby('topic')

# sort groups by size
grouped = sorted(grouped, key=lambda x: len(x[1]), reverse=True)

# assign group numbers back to stories
for i, (name, group) in enumerate(grouped):
    # TODO: I THINK THIS IS BROKEN 🐛, getting weird items into
    stories.loc[stories['topic'] == name, 'topic'] = name

print("Number of groups:", len(grouped))
# Number of items in each group
print("Group sizes:")
print([len(group) for name, group in grouped])



In [32]:
def summarize_topic(titles):
    """
    Pass list of titles to ChatGPT and ask it to summarize them in 2-4 words.
    """

    # Combine the titles into a single string
    titles_str = ', '.join(titles)

    # print("Writing a title for")
    # for title in titles[:5]:
    #     print(f"  - {title}")
    
    MODEL = "gpt-3.5-turbo"
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"The following article titles form a topic. \n\n {titles_str} \n\n Please write a specific summary of the topic in 2-4 words:"},
        ],
        max_tokens=10
    )

    return response.choices[0].message.content

In [ ]:
# make a list of titles per topic
topic_titles = stories.groupby('topic')['title'].apply(list).to_dict()
topic_titles = [{
    'topic': k,
    'num_articles': len(v),
    'headlines': v
} for k,v in topic_titles.items()]

# sort by num_articles
topic_titles = sorted(topic_titles, key=lambda x: x['num_articles'], reverse=True)

# pass each topic list of titles to openai chatgpt and ask it to summarize the topic in 2-4 words
for topic in topic_titles[0:]:
    print(f"Topic {topic['topic']} ({topic['num_articles']} articles)")
    
    if topic['topic'] == -1:
        continue

    # if there are more than 10 articles in a topic, sample 10 (to keep within the word limit of the API)
    if topic['num_articles'] >= 10:
        headlines = np.random.choice(topic['headlines'], 10, replace=False)
        # Summary
        try:
            topic['topic_summary'] = summarize_topic(headlines)
            print(topic['topic_summary'])
        except InvalidRequestError:
            topic['topic_summary'] = "Error Making Summary From OpenAI API"
            print("OpenAI API request failed.")
    else:
        headlines = topic['headlines']

In [40]:
# turn topic titles and summaries into a dataframe
topic_titles_df = pd.DataFrame(topic_titles)
topic_titles_df = topic_titles_df[['topic', 'topic_summary']]
stories = stories.merge(topic_titles_df, on='topic', how='left')

In [50]:
stories.to_csv('../stories-with-embeddings.csv', index=False)


# Collect Metadata

In [ ]:
# loop through topic_titles
topic_metadata = []
for topic in topic_titles:
    # grab topic, num_articles, and summary only
    topic = {k:v for k,v in topic.items() if k in ['topic', 'num_articles', 'topic_summary']}
    topic_metadata.append(topic)

topic_metadata

In [ ]:
# read output/metadata.json
import json
with open('output/metadata.json') as f:
    metadata = json.load(f)

metadata['topics'] = topic_metadata

# write metadata back to json file
with open('output/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

metadata

In [44]:
# collect top keywords
top_keywords = stories\
    .explode('keywords')\
    .groupby('keywords')\
    .size()\
    .reset_index(name='count')\
    .sort_values(by='count', ascending=False)\
    .head(100)
    

In [ ]:
stories.query("title.str.contains('Harris') & ~title.str.contains('Trump')")